# 05 — Tester PitchLens sur une vidéo

Ce notebook applique un modèle entraîné à **chaque image d'une vidéo**, dessine les détections du ballon, puis crée une nouvelle vidéo MP4.

Il ne charge jamais toute la vidéo en mémoire : une image est lue, analysée et écrite à la fois. On peut donc l'utiliser sur une vidéo de plusieurs minutes sans reproduire le problème de mémoire rencontré avec les grandes galeries d'images.

> Attendre la fin de l'entraînement avant de lancer ce notebook. Le fichier `best.pt` peut encore changer pendant l'entraînement.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import time

import cv2
import torch
from IPython.display import Markdown, Video, display
from tqdm.auto import tqdm
from ultralytics import YOLO

# Fonctionne si Jupyter a été ouvert depuis la racine du projet ou depuis notebooks/.
PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (folder / "pyproject.toml").exists()
)
print(f"Projet : {PROJECT_ROOT}")

## 1. Paramètres

Les deux réglages indispensables sont :

- `VIDEO_PATH` : chemin vers la vidéo à analyser ;
- `MODEL_PATH` : `None` choisit automatiquement le `best.pt` le plus récent. On peut aussi fournir un chemin précis.

`CONFIDENCE = 0.25` signifie qu'une prédiction est dessinée à partir de 25 % de confiance. Augmenter ce seuil réduit les fausses alertes, mais risque de rater davantage de ballons.

In [ ]:
# À MODIFIER : déposer la vidéo où tu veux, puis indiquer son chemin ici.
VIDEO_PATH = PROJECT_ROOT / "data/external/videos/ma_video.mp4"

# None = dernier best.pt produit dans models/checkpoints/baseline_runs/.
MODEL_PATH = None

CONFIDENCE = 0.25
IMAGE_SIZE = 640
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
OUTPUT_MAX_WIDTH = 1920  # réduit uniquement la vidéo de sortie si la source est en 4K
MAX_FRAMES = None        # ex. 300 pour tester seulement les 300 premières images

OUTPUT_DIR = PROJECT_ROOT / "models/reports/video_inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_PATH is None:
    candidates = sorted(
        (PROJECT_ROOT / "models/checkpoints/baseline_runs").glob("*/train/weights/best.pt"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError("Aucun best.pt trouvé. Termine d'abord un entraînement.")
    MODEL_PATH = candidates[0]
else:
    MODEL_PATH = Path(MODEL_PATH).expanduser().resolve()

VIDEO_PATH = Path(VIDEO_PATH).expanduser().resolve()
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Vidéo introuvable : {VIDEO_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Modèle introuvable : {MODEL_PATH}")

print(f"Vidéo  : {VIDEO_PATH}")
print(f"Modèle : {MODEL_PATH}")
print(f"Calcul : {DEVICE}")

## 2. Lire les caractéristiques de la vidéo

Cette étape ne lance pas encore le modèle. Elle permet de vérifier la durée, le nombre d'images par seconde et la résolution.

In [ ]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f"OpenCV ne peut pas ouvrir : {VIDEO_PATH}")

FPS = capture.get(cv2.CAP_PROP_FPS) or 30.0
FRAME_COUNT = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
SOURCE_WIDTH = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
SOURCE_HEIGHT = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.release()

frames_to_process = min(FRAME_COUNT, MAX_FRAMES) if MAX_FRAMES else FRAME_COUNT
duration_seconds = frames_to_process / FPS
display(Markdown(
    f"**Source :** {SOURCE_WIDTH} × {SOURCE_HEIGHT} — {FPS:.2f} images/s — "
    f"{frames_to_process:,} images à traiter — {duration_seconds / 60:.1f} minute(s)."
))

## 3. Détecter le ballon et créer la vidéo

Pour chaque image :

1. YOLO redimensionne temporairement l'image à `IMAGE_SIZE=640` pour l'inférence ;
2. les coordonnées prédites sont replacées sur l'image d'origine ;
3. un rectangle rouge et la confiance sont dessinés ;
4. l'image annotée est immédiatement écrite dans la vidéo de sortie.

Le son original est ensuite recopié avec `ffmpeg` lorsqu'il est disponible.

In [ ]:
model = YOLO(str(MODEL_PATH))

scale = min(1.0, OUTPUT_MAX_WIDTH / SOURCE_WIDTH)
output_width = int(SOURCE_WIDTH * scale)
output_height = int(SOURCE_HEIGHT * scale)
# Certains codecs exigent des dimensions paires.
output_width -= output_width % 2
output_height -= output_height % 2

run_name = MODEL_PATH.parent.parent.parent.name
SILENT_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}__{run_name}__silent.mp4"
FINAL_VIDEO = OUTPUT_DIR / f"{VIDEO_PATH.stem}__{run_name}__ball.mp4"

capture = cv2.VideoCapture(str(VIDEO_PATH))
writer = cv2.VideoWriter(
    str(SILENT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (output_width, output_height),
)
if not writer.isOpened():
    capture.release()
    raise RuntimeError("Impossible de créer la vidéo temporaire avec le codec mp4v.")

started_at = time.perf_counter()
processed = 0
frames_with_detection = 0
detections = 0

try:
    with tqdm(total=frames_to_process, unit="image", desc="Inférence vidéo") as progress:
        while processed < frames_to_process:
            ok, frame = capture.read()
            if not ok:
                break

            result = model.predict(
                frame,
                imgsz=IMAGE_SIZE,
                conf=CONFIDENCE,
                device=DEVICE,
                verbose=False,
            )[0]

            boxes = result.boxes
            if boxes is not None and len(boxes):
                frames_with_detection += 1
                detections += len(boxes)
                for xyxy, confidence in zip(boxes.xyxy.cpu().numpy(), boxes.conf.cpu().numpy()):
                    x1, y1, x2, y2 = (int(value) for value in xyxy)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                    label = f"ball {confidence:.2f}"
                    text_y = max(22, y1 - 8)
                    cv2.putText(
                        frame, label, (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, (0, 0, 255), 2, cv2.LINE_AA,
                    )

            if (output_width, output_height) != (SOURCE_WIDTH, SOURCE_HEIGHT):
                frame = cv2.resize(frame, (output_width, output_height), interpolation=cv2.INTER_AREA)
            writer.write(frame)
            processed += 1
            progress.update(1)
finally:
    capture.release()
    writer.release()

# Réencodage H.264 lisible dans la plupart des lecteurs + récupération éventuelle du son.
ffmpeg = shutil.which("ffmpeg")
if ffmpeg:
    subprocess.run(
        [
            ffmpeg, "-y", "-loglevel", "error",
            "-i", str(SILENT_VIDEO), "-i", str(VIDEO_PATH),
            "-map", "0:v:0", "-map", "1:a?",
            "-c:v", "libx264", "-preset", "fast", "-crf", "20",
            "-c:a", "aac", "-shortest", str(FINAL_VIDEO),
        ],
        check=True,
    )
    SILENT_VIDEO.unlink()
else:
    if FINAL_VIDEO.exists():
        FINAL_VIDEO.unlink()
    SILENT_VIDEO.rename(FINAL_VIDEO)

elapsed = time.perf_counter() - started_at
display(Markdown(
    f"### Terminé\n"
    f"- **Temps :** {elapsed / 60:.1f} minute(s)\n"
    f"- **Vitesse moyenne :** {processed / elapsed:.1f} images/s\n"
    f"- **Images avec au moins une détection :** {frames_with_detection:,} / {processed:,}\n"
    f"- **Nombre total de boîtes dessinées :** {detections:,}\n"
    f"- **Fichier :** `{FINAL_VIDEO}`"
))

## 4. Regarder le résultat

La vidéo reste stockée comme un fichier séparé : elle n'est pas intégrée dans le notebook, afin de ne pas rendre le fichier `.ipynb` énorme.

In [ ]:
display(Video(filename=str(FINAL_VIDEO), embed=False, width=960))